# 01 — Data Cleaning and Repeatability Preparation

This notebook turns the raw measurement files into a clean, reproducible dataset.

Outputs:
- `../data/processed/resistance_sweep_run1_clean.csv`
- `../data/processed/resistance_sweep_run2_clean.csv`
- `../data/processed/resistance_sweep_combined_clean.csv`
- `../data/processed/repeatability_pointwise_comparison.csv`
- `../data/processed/repeatability_summary.csv`
- figures in `../figures/repeatability/`

The key rule: **do not trust manually typed Gain**. Recompute it from `VR_Vpp / Vs_Vpp`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RUN1_PATH = Path("../data/raw/resistance_sweep_run1.csv")
RUN2_PATH = Path("../data/raw/resistance_sweep_run2.csv")

PROCESSED_DIR = Path("../data/processed")
FIG_DIR = Path("../figures/repeatability")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 12,
    "axes.grid": True,
})

In [ ]:
def load_and_standardize(path, run_number):
    """Load a raw resistance-sweep CSV and standardize it."""

    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()

    # Accept common column variants.
    df = df.rename(columns={
        "R": "Resistance",
        "Resistance_Ohm": "Resistance",
        "Frequency": "Frequency_Hz",
        "Freq": "Frequency_Hz",
        "Vs": "Vs_Vpp",
        "VR": "VR_Vpp",
        "Vr": "VR_Vpp",
    })

    required = ["Resistance", "Frequency_Hz", "Vs_Vpp", "VR_Vpp"]
    missing = [col for col in required if col not in df.columns]

    if missing:
        raise ValueError(
            f"{path} is missing columns {missing}. "
            f"Current columns: {df.columns.tolist()}"
        )

    df = df[required].copy()

    for col in required:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    before = len(df)
    df = df.dropna(subset=required)
    after = len(df)

    if before != after:
        print(f"{path.name}: dropped {before - after} rows with non-numeric/missing values.")

    df["Gain"] = df["VR_Vpp"] / df["Vs_Vpp"]
    df["Run"] = run_number

    df = df.sort_values(["Resistance", "Frequency_Hz"]).reset_index(drop=True)
    return df

In [ ]:
run1 = load_and_standardize(RUN1_PATH, run_number=1)
run2 = load_and_standardize(RUN2_PATH, run_number=2)

display(run1.head())
display(run2.head())

print("Run 1 shape:", run1.shape)
print("Run 2 shape:", run2.shape)

In [ ]:
run1.to_csv(PROCESSED_DIR / "resistance_sweep_run1_clean.csv", index=False)
run2.to_csv(PROCESSED_DIR / "resistance_sweep_run2_clean.csv", index=False)

combined = pd.concat([run1, run2], ignore_index=True)
combined = combined.sort_values(["Resistance", "Run", "Frequency_Hz"]).reset_index(drop=True)

combined.to_csv(PROCESSED_DIR / "resistance_sweep_combined_clean.csv", index=False)

combined.head()

## Repeatability comparison

This compares Run 1 and Run 2 only at matched `(Resistance, Frequency_Hz)` points.

In [ ]:
comparison = pd.merge(
    run1,
    run2,
    on=["Resistance", "Frequency_Hz"],
    suffixes=("_Run1", "_Run2"),
    how="inner"
)

print(f"Run 1 rows: {len(run1)}")
print(f"Run 2 rows: {len(run2)}")
print(f"Matched rows: {len(comparison)}")

comparison.head()

In [ ]:
comparison["Gain_Diff"] = comparison["Gain_Run2"] - comparison["Gain_Run1"]
comparison["Abs_Gain_Diff"] = comparison["Gain_Diff"].abs()

comparison["Relative_Gain_Diff_Percent"] = (
    100 * comparison["Abs_Gain_Diff"] / comparison["Gain_Run1"].abs()
)

comparison["Vs_Diff"] = comparison["Vs_Vpp_Run2"] - comparison["Vs_Vpp_Run1"]
comparison["VR_Diff"] = comparison["VR_Vpp_Run2"] - comparison["VR_Vpp_Run1"]

relative_threshold_percent = 8
absolute_threshold = 0.05

comparison["Suspicious"] = (
    (comparison["Relative_Gain_Diff_Percent"] > relative_threshold_percent)
    | (comparison["Abs_Gain_Diff"] > absolute_threshold)
)

comparison.head()

In [ ]:
summary = (
    comparison
    .groupby("Resistance")
    .agg(
        Mean_Abs_Gain_Diff=("Abs_Gain_Diff", "mean"),
        Max_Abs_Gain_Diff=("Abs_Gain_Diff", "max"),
        Std_Gain_Diff=("Gain_Diff", "std"),
        Mean_Relative_Diff_Percent=("Relative_Gain_Diff_Percent", "mean"),
        Max_Relative_Diff_Percent=("Relative_Gain_Diff_Percent", "max"),
        Num_Points=("Gain_Diff", "count"),
        Num_Suspicious=("Suspicious", "sum"),
    )
    .reset_index()
)

comparison.to_csv(PROCESSED_DIR / "repeatability_pointwise_comparison.csv", index=False)
summary.to_csv(PROCESSED_DIR / "repeatability_summary.csv", index=False)

summary

In [ ]:
# Run 1 vs Run 2 gain curves for each resistance
for R in sorted(comparison["Resistance"].unique()):
    subset = comparison[comparison["Resistance"] == R].sort_values("Frequency_Hz")

    fig, ax = plt.subplots(figsize=(9, 5))

    ax.semilogx(subset["Frequency_Hz"], subset["Gain_Run1"], "o-", label="Run 1", markersize=4)
    ax.semilogx(subset["Frequency_Hz"], subset["Gain_Run2"], "s--", label="Run 2", markersize=4)

    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Gain |VR/Vs|")
    ax.set_title(f"Repeatability Check: R = {R:.0f} Ω")
    ax.grid(True, which="both", alpha=0.35)
    ax.legend()

    fig.savefig(FIG_DIR / f"gain_repeatability_R_{int(R)}ohm.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

for R, group in comparison.groupby("Resistance"):
    group = group.sort_values("Frequency_Hz")
    ax.semilogx(group["Frequency_Hz"], group["Abs_Gain_Diff"], "o-", markersize=4, label=f"{R:.0f} Ω")

ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Absolute Gain Difference")
ax.set_title("Pointwise Repeatability Error vs Frequency")
ax.grid(True, which="both", alpha=0.35)
ax.legend(title="Resistance")

fig.savefig(FIG_DIR / "absolute_gain_difference_vs_frequency.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(summary["Resistance"], summary["Mean_Abs_Gain_Diff"], "o-", linewidth=2)

ax.set_xlabel("Resistance (Ω)")
ax.set_ylabel("Mean Absolute Gain Difference")
ax.set_title("Mean Repeatability Error vs Resistance")
ax.grid(True, alpha=0.35)

fig.savefig(FIG_DIR / "mean_absolute_gain_difference_vs_resistance.png", dpi=300, bbox_inches="tight")
plt.show()

## Interpretation

If repeatability error is much smaller than model residuals, model inadequacy is probably the limiting factor.

If repeatability error is comparable to model residuals, then the data quality limits the reliability of parameter reconstruction.

Suspicious points are flagged, not deleted. Deleting data silently is bad scientific practice.